# §10 Sensitivity + Oracle + Visualization + Baselines

Drop-in analysis after §9. Reads the 24h / 48h prediction CSVs and tests:

1. **Prediction perturbation** — δ ∈ {±10%, ±30%}, two schemes
2. **Blend weight sweep** — `w ∈ {0, 0.25, 0.5, 0.75, 1.0}`
3. **Oracle comparison** — regret = `(oracle_mit − our_mit) / oracle_mit`
4. **Visualization** — 48h heatmap, top-10 forecast curves with generator markers, greedy gain curve
5. **Alternative decisions** — population & historical-mean baselines vs our data-driven greedy

**Inputs:** `results/direct_pred_24h.csv`, `results/direct_pred_48h.csv` (from §8)

**Outputs:**
- `results/sensitivity_heatmap.png`, `results/sensitivity_county_freq.csv`
- `results/uniform_picks_detail.csv`, `results/blend_sweep_picks.csv`
- `results/regret_scenarios.png`, `results/regret_scenarios.csv`
- `results/pred_48h_heatmap.png`
- `results/top10_curves_with_gens.png`
- `results/greedy_gain_curve.png`
- `results/baseline_comparison.png`, `results/baseline_comparison.csv`


In [ ]:
# ---- Config ----
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter

NUM_GENERATORS = 5
GEN_CAPACITY   = 1000
N_TRIALS       = 100              # Monte-Carlo trials per (delta, w)
RNG            = np.random.default_rng(42)

PRED_24H_CSV = 'results/direct_pred_24h.csv'
PRED_48H_CSV = 'results/direct_pred_48h.csv'

os.makedirs('results', exist_ok=True)

## 1. Load predictions from CSV, pivot to (T, L) matrices

In [ ]:
def load_pred_csv(path, expected_T):
    df = pd.read_csv(path)
    assert set(df.columns) >= {'timestamp', 'location', 'pred'}, \
        f"{path} 列不对: {df.columns.tolist()}"
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['location']  = df['location'].astype(str)
    mat_df = df.pivot(index='timestamp', columns='location', values='pred')
    mat_df = mat_df.sort_index()
    mat_df = mat_df.reindex(sorted(mat_df.columns), axis=1)
    assert mat_df.shape[0] == expected_T, \
        f"{path} 时间步 {mat_df.shape[0]} != 期望 {expected_T}"
    assert not mat_df.isna().any().any(), f"{path} 有 NaN"
    return mat_df.values.astype(np.float64), list(mat_df.columns), list(mat_df.index)

pred_24h_mat, locs_24, ts_24 = load_pred_csv(PRED_24H_CSV, expected_T=24)
pred_48h_mat, locs_48, ts_48 = load_pred_csv(PRED_48H_CSV, expected_T=48)

assert locs_24 == locs_48, "24h/48h CSV county 列表不一致"
locs = locs_48
L    = len(locs)
print(f"pred_24h_mat: {pred_24h_mat.shape} (peak={pred_24h_mat.max():.1f})")
print(f"pred_48h_mat: {pred_48h_mat.shape} (peak={pred_48h_mat.max():.1f})")
print(f"县数: {L}, 前 24h 时间窗: {ts_48[0]} .. {ts_48[23]}")

## 2. Greedy allocator + blend + overlap metrics

In [ ]:
def greedy_allocate(pred_mat, n_gen=NUM_GENERATORS, cap=GEN_CAPACITY,
                    return_gains=False):
    """§9 贪心分配器的向量化版本."""
    L_ = pred_mat.shape[1]
    alloc = np.zeros(L_, dtype=int)
    picks, step_gains = [], []
    for _ in range(n_gen):
        cur_caps = (alloc      * cap)[None, :]
        nxt_caps = ((alloc+1) * cap)[None, :]
        gains = (np.minimum(pred_mat, nxt_caps).sum(axis=0)
                 - np.minimum(pred_mat, cur_caps).sum(axis=0))
        best_i = int(np.argmax(gains))
        alloc[best_i] += 1
        picks.append(best_i)
        step_gains.append(float(gains[best_i]))
    return (picks, step_gains) if return_gains else picks

def build_blended(w, p24, p48):
    out = p48.copy()
    out[:24] = w * p24 + (1.0 - w) * p48[:24]
    return np.clip(out, 0, None)

def multiset_overlap(a, b):
    ca, cb = Counter(a), Counter(b)
    return sum(min(ca[k], cb[k]) for k in ca)

## 3. Baseline decision (w = 0.5, no noise)

In [ ]:
base_blended   = build_blended(0.5, pred_24h_mat, pred_48h_mat)
baseline_picks = greedy_allocate(base_blended)
baseline_fips  = [locs[i] for i in baseline_picks]
print(f"Baseline (w=0.5, no noise): {baseline_fips}")

deltas = [-0.30, -0.10, 0.0, 0.10, 0.30]
ws     = [0.0, 0.25, 0.5, 0.75, 1.0]

## 4. Scan A — Stochastic (multiplicative Gaussian noise)

In [ ]:
mean_overlap = np.zeros((len(deltas), len(ws)))
exact_match  = np.zeros((len(deltas), len(ws)))
county_freq  = Counter()
total_trials = 0

for i, delta in enumerate(deltas):
    for j, w in enumerate(ws):
        blended = build_blended(w, pred_24h_mat, pred_48h_mat)
        sigma   = abs(delta) + 0.05
        overlaps, exacts = [], []
        for _ in range(N_TRIALS):
            noise     = RNG.normal(1.0 + delta, sigma, size=blended.shape)
            perturbed = np.clip(blended * noise, 0, None)
            picks     = greedy_allocate(perturbed)
            ov        = multiset_overlap(picks, baseline_picks)
            overlaps.append(ov)
            exacts.append(int(ov == NUM_GENERATORS))
            for idx in picks:
                county_freq[locs[idx]] += 1
            total_trials += 1
        mean_overlap[i, j] = np.mean(overlaps)
        exact_match[i, j]  = np.mean(exacts)

print(f"完成 Scan A: {total_trials} 次随机 trials")

## 5. Scan B — Deterministic uniform scaling

In [ ]:
uniform_overlap = np.zeros((len(deltas), len(ws)), dtype=int)
uniform_picks_table = []
for i, delta in enumerate(deltas):
    for j, w in enumerate(ws):
        blended = build_blended(w, pred_24h_mat, pred_48h_mat) * (1.0 + delta)
        blended = np.clip(blended, 0, None)
        picks   = greedy_allocate(blended)
        uniform_overlap[i, j] = multiset_overlap(picks, baseline_picks)
        uniform_picks_table.append({
            'delta'  : delta, 'w': w,
            'picks'  : [locs[p] for p in picks],
            'overlap': uniform_overlap[i, j],
        })

uniform_df = pd.DataFrame(uniform_picks_table)
uniform_df.to_csv('results/uniform_picks_detail.csv', index=False)

changed = uniform_df[uniform_df['overlap'] != NUM_GENERATORS]
print("--- Cells where decision DIFFERS from baseline ---")
print(changed.to_string(index=False) if len(changed) else "  (none)")
print(f"\nTotal: {len(uniform_df)}, changed: {len(changed)}")

## 6. Sensitivity heatmap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def _heatmap(ax, mat, title, vmin, vmax, cmap, fmt=".2f"):
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(ws)));     ax.set_xticklabels([f"w={w}" for w in ws])
    ax.set_yticks(range(len(deltas))); ax.set_yticklabels([f"{int(d*100):+d}%" for d in deltas])
    ax.set_xlabel("Blend weight w"); ax.set_ylabel("Prediction perturbation δ")
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            color = 'white' if (v - vmin) / (vmax - vmin + 1e-9) < 0.45 else 'black'
            ax.text(j, i, format(v, fmt), ha='center', va='center', color=color, fontsize=9)
    return im

im1 = _heatmap(axes[0], mean_overlap, f"Mean overlap @ 5 (noisy, N={N_TRIALS})", 0, 5, 'viridis')
plt.colorbar(im1, ax=axes[0])
im2 = _heatmap(axes[1], exact_match, "P(exact multiset match)", 0, 1, 'RdYlGn')
plt.colorbar(im2, ax=axes[1])
im3 = _heatmap(axes[2], uniform_overlap.astype(float),
               "Overlap @ 5 (uniform scaling)", 0, 5, 'viridis', fmt=".0f")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.savefig('results/sensitivity_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. County robustness table

In [ ]:
freq_df = pd.DataFrame([
    {'FIPS': fips, 'count': cnt,
     'freq_%': 100.0 * cnt / total_trials,
     'in_baseline': fips in baseline_fips}
    for fips, cnt in county_freq.most_common()
])
freq_df.to_csv('results/sensitivity_county_freq.csv', index=False)
print(freq_df.head(15).to_string(index=False))

## 8. Pure blend sweep (no noise)

In [ ]:
blend_rows = []
for w in ws:
    blended = build_blended(w, pred_24h_mat, pred_48h_mat)
    picks   = greedy_allocate(blended)
    fips    = [locs[p] for p in picks]
    ov      = multiset_overlap(picks, baseline_picks)
    blend_rows.append({'w': w, 'picks': fips, 'overlap_vs_baseline': ov})
    print(f"  w={w:.2f}: {fips}  (overlap={ov}/5)")

pd.DataFrame(blend_rows).to_csv('results/blend_sweep_picks.csv', index=False)

## 9. Oracle comparison — Decision regret

**Regret:** $(\text{oracle\_mit} - \text{our\_mit}) / \text{oracle\_mit} \in [0, 1]$


In [ ]:
def mitigation_total(truth, picks, cap=GEN_CAPACITY):
    L_ = truth.shape[1]
    counts = np.zeros(L_, dtype=int)
    for p in picks:
        counts[p] += 1
    caps = (counts * cap)[None, :]
    return float(np.minimum(truth, caps).sum())

def compute_regret(truth, our_picks):
    our_mit = mitigation_total(truth, our_picks)
    oracle_picks = greedy_allocate(truth)
    oracle_mit = mitigation_total(truth, oracle_picks)
    if oracle_mit < 1e-6:
        return {'our_mit': our_mit, 'oracle_mit': oracle_mit, 'regret': 0.0,
                'oracle_picks': [locs[p] for p in oracle_picks]}
    return {'our_mit': our_mit, 'oracle_mit': oracle_mit,
            'regret': (oracle_mit - our_mit) / oracle_mit,
            'oracle_picks': [locs[p] for p in oracle_picks]}

### 9.1 Demo test scenario (skips if .nc not available)


In [ ]:
demo_test_truth = None
DEMO_TEST_CANDIDATES = [
    'data/test_48h_demo.nc',
    'drive/MyDrive/test_48h_demo.nc',
    '/content/drive/MyDrive/MLPS_Final_Project/data/test_48h_demo.nc',
]

try:
    import xarray as xr
    for p in DEMO_TEST_CANDIDATES:
        if not os.path.exists(p):
            continue
        ds_dt = xr.open_dataset(p)
        if not hasattr(ds_dt, 'out'):
            continue
        truth_arr = ds_dt.out.transpose('timestamp', 'location').values.astype(np.float64)
        ds_locs = [str(l) for l in ds_dt.location.values]
        if set(ds_locs) >= set(locs):
            idx_map = [ds_locs.index(l) for l in locs]
            demo_test_truth = np.clip(truth_arr[:, idx_map], 0, None)
            print(f"Loaded demo test from {p}")
            break
except ImportError:
    print("[skip] xarray not installed")

if demo_test_truth is not None and demo_test_truth.shape[0] >= 48:
    res = compute_regret(demo_test_truth[:48], baseline_picks)
    print(f"\n[demo test, noise]: our={res['our_mit']:.1f}, "
          f"oracle={res['oracle_mit']:.1f}, regret={res['regret']:.2%}")
else:
    print("[skip] demo test .nc 不可用 — 直接进入 synthetic 场景")

### 9.2 Synthetic storm scenarios

In [ ]:
def make_scenarios(pred_24h, pred_48h, rng, n_per_type=5):
    base = build_blended(0.5, pred_24h, pred_48h)
    L_ = base.shape[1]
    scenarios = [('Forecast-as-truth', base.copy())]

    for k in range(n_per_type):
        scenarios.append((f'Noisy ±20% #{k+1}',
                         np.clip(base * rng.normal(1.0, 0.20, base.shape), 0, None)))

    totals = base.sum(axis=0)
    top5 = np.argsort(-totals)[:5]
    pool = np.array([i for i in range(L_) if i not in top5])
    for k in range(n_per_type):
        truth = base.copy()
        targets = rng.choice(pool, 5, replace=False)
        for a, b in zip(top5, targets):
            tmp = truth[:, a].copy()
            truth[:, a] = truth[:, b]
            truth[:, b] = tmp
        scenarios.append((f'Epicenter shift #{k+1}', truth))

    for factor in [0.5, 0.7, 1.3, 1.5, 2.0]:
        scenarios.append((f'Intensity {factor:.1f}x', np.clip(base * factor, 0, None)))

    for shift in [-6, -3, 3, 6]:
        truth = np.roll(base, shift, axis=0)
        if shift > 0: truth[:shift] = 0
        else:         truth[shift:] = 0
        scenarios.append((f'Timing shift {shift:+d}h', truth))

    return scenarios

scenarios = make_scenarios(pred_24h_mat, pred_48h_mat, RNG, n_per_type=5)
regret_rows = []
for name, truth in scenarios:
    res = compute_regret(truth, baseline_picks)
    regret_rows.append({
        'scenario': name, 'our_mit': res['our_mit'],
        'oracle_mit': res['oracle_mit'], 'regret_%': res['regret'] * 100,
        'oracle_picks': res['oracle_picks'],
    })
regret_df = pd.DataFrame(regret_rows)
regret_df.to_csv('results/regret_scenarios.csv', index=False)
print(regret_df[['scenario', 'our_mit', 'oracle_mit', 'regret_%']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(6, 0.32 * len(regret_df))))
df_sorted = regret_df.sort_values('regret_%').reset_index(drop=True)
colors = ['#27ae60' if r < 5 else '#f39c12' if r < 20 else '#c0392b'
          for r in df_sorted['regret_%']]
bars = ax.barh(df_sorted['scenario'], df_sorted['regret_%'], color=colors)
for bar, val in zip(bars, df_sorted['regret_%']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f"{val:.1f}%",
            va='center', fontsize=8)
ax.set_xlabel('Regret (%)  —  lower = better')
ax.set_title(f'Decision regret across {len(regret_df)} scenarios')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('results/regret_scenarios.png', dpi=120, bbox_inches='tight')
plt.show()

def family(name):
    if 'Forecast' in name:    return 'Forecast-as-truth'
    if 'Noisy' in name:       return 'Noisy ±20%'
    if 'Epicenter' in name:   return 'Epicenter shift'
    if 'Intensity' in name:   return 'Intensity surprise'
    if 'Timing' in name:      return 'Timing shift'
    return 'Other'

regret_df['family'] = regret_df['scenario'].apply(family)
print(regret_df.groupby('family')['regret_%'].agg(['mean', 'min', 'max']).round(2).to_string())

## 10. Visualizations

Three exploratory plots to support the report:

- **10.1** 48h prediction heatmap (timestamp × county) — shows storm spatiotemporal footprint
- **10.2** Top-10 county forecast curves with generator markers — visual evidence of allocation
- **10.3** Greedy gain curve — diminishing-returns analysis of marginal mitigation per generator


### 10.1 48h prediction heatmap (counties sorted by total outage)

In [ ]:
totals = base_blended.sum(axis=0)
order  = np.argsort(-totals)
mat_sorted  = base_blended[:, order]
locs_sorted = [locs[i] for i in order]

fig, ax = plt.subplots(figsize=(14, 11))
norm = mcolors.SymLogNorm(linthresh=10, vmin=0,
                           vmax=max(mat_sorted.max(), 100), base=10)
im = ax.imshow(mat_sorted.T, aspect='auto', cmap='YlOrRd', norm=norm,
               interpolation='nearest')
cbar = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Predicted outage count (symlog scale)')

# Highlight the rows that received generators
gen_count_by_fips = Counter(baseline_fips)
for rank, fips in enumerate(locs_sorted):
    n = gen_count_by_fips.get(fips, 0)
    if n > 0:
        ax.axhspan(rank - 0.5, rank + 0.5, fill=False,
                   edgecolor='#1565C0', linewidth=2)
        ax.text(48.5, rank, f"  {fips} ({n} gen)", va='center',
                fontsize=8, color='#1565C0', fontweight='bold')

ax.set_xlabel('Hour (0 = first hour of forecast window)')
ax.set_ylabel(f'County (sorted by total predicted outage, top = highest)')
ax.set_title(f'48h prediction heatmap, all {L} counties\n'
             f'Boxes mark generator-receiving counties')

# Show ticks every 5 counties
n_ticks = min(20, L)
tick_idx = np.linspace(0, L-1, n_ticks).astype(int)
ax.set_yticks(tick_idx)
ax.set_yticklabels([locs_sorted[i] for i in tick_idx], fontsize=7)
ax.set_xticks(np.arange(0, 49, 6))

plt.tight_layout()
plt.savefig('results/pred_48h_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Top-3 by total: {[locs_sorted[i] for i in range(3)]} "
      f"(totals: {[int(totals[order[i]]) for i in range(3)]})")

### 10.2 Top-10 county forecast curves + generator placement

The 3 generator-receiving counties are highlighted; horizontal lines show 1× and 2× capacity.
The shaded region above 1,000 is the part of an outage *not* covered by 1 generator.


In [ ]:
top10_idx = np.argsort(-base_blended.sum(axis=0))[:10]
hours = np.arange(48)

fig, ax = plt.subplots(figsize=(13, 7))
gen_count_by_fips = Counter(baseline_fips)

# Draw capacity reference lines first (behind data)
for k_caps in [1, 2]:
    ax.axhline(k_caps * GEN_CAPACITY, color='#c0392b',
               linestyle='--' if k_caps == 1 else ':',
               alpha=0.5, linewidth=1.0,
               label=f'{k_caps}-gen capacity ({k_caps*GEN_CAPACITY})')

palette = plt.cm.tab10(np.linspace(0, 1, 10))
for rank, i in enumerate(top10_idx):
    fips = locs[i]
    n_gen = gen_count_by_fips.get(fips, 0)
    if n_gen > 0:
        ax.plot(hours, base_blended[:, i], linewidth=2.8,
                color=palette[rank],
                label=f"{fips} ({n_gen} gen) ★", alpha=1.0, zorder=5)
        # Mark peak
        peak_t = int(np.argmax(base_blended[:, i]))
        peak_v = base_blended[peak_t, i]
        ax.scatter([peak_t], [peak_v], s=80, color=palette[rank],
                   edgecolor='black', linewidth=1.5, zorder=6)
    else:
        ax.plot(hours, base_blended[:, i], linewidth=1.2,
                color=palette[rank], label=fips, alpha=0.55, zorder=2)

ax.set_xlabel('Hour')
ax.set_ylabel('Predicted outage count')
ax.set_title(f'Top-10 county 48h forecasts; ★ = receives generator\n'
             f'Allocation: {dict(gen_count_by_fips)}')
ax.legend(loc='upper right', fontsize=9, framealpha=0.92)
ax.grid(alpha=0.3)
ax.set_xticks(np.arange(0, 49, 6))

plt.tight_layout()
plt.savefig('results/top10_curves_with_gens.png', dpi=120, bbox_inches='tight')
plt.show()

### 10.3 Greedy gain curve — diminishing returns

Two views:
- **Left**: marginal mitigation gained from each additional generator (strictly decreasing for greedy)
- **Right**: cumulative mitigation as generators are added, expressed as % of total predicted outage

A steep early curve that flattens means the first few generators capture most of the value —
useful for arguing why 5 generators is enough vs. requesting more.


In [ ]:
picks_g, gains_g = greedy_allocate(base_blended, return_gains=True)
cumulative = np.cumsum(gains_g)
total_predicted = base_blended.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: marginal gain per generator
bars = axes[0].bar(range(1, NUM_GENERATORS + 1), gains_g,
                   color=['#1565C0', '#1976D2', '#2196F3', '#64B5F6', '#90CAF9'])
for i, (g, p) in enumerate(zip(gains_g, picks_g)):
    fips = locs[p]
    axes[0].text(i + 1, g, f'{int(g):,}\n→{fips}',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Generator #')
axes[0].set_ylabel('Marginal mitigation gain (outage-hours)')
axes[0].set_title('Marginal benefit per generator (greedy)')
axes[0].set_xticks(range(1, NUM_GENERATORS + 1))
axes[0].grid(axis='y', alpha=0.3)

# Right: cumulative coverage
axes[1].plot(range(1, NUM_GENERATORS + 1), cumulative,
             marker='o', markersize=10, linewidth=2.5, color='#1565C0')
axes[1].axhline(total_predicted, color='gray', linestyle='--', alpha=0.5,
                label=f'Total predicted outage ({int(total_predicted):,})')
for i, c in enumerate(cumulative):
    pct = 100 * c / total_predicted
    axes[1].text(i + 1, c, f'  {pct:.1f}%', ha='left', va='center', fontsize=10)
axes[1].set_xlabel('Generators deployed')
axes[1].set_ylabel('Cumulative mitigation (outage-hours)')
axes[1].set_title('Cumulative coverage')
axes[1].set_xticks(range(1, NUM_GENERATORS + 1))
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, total_predicted * 1.05)

plt.tight_layout()
plt.savefig('results/greedy_gain_curve.png', dpi=120, bbox_inches='tight')
plt.show()

# Print decay analysis
print("Marginal gain decay analysis:")
for i in range(NUM_GENERATORS):
    pct_total = 100 * cumulative[i] / total_predicted
    if i == 0:
        print(f"  Gen #1: {gains_g[i]:>9.0f} ({pct_total:.1f}% of total)")
    else:
        retention = 100 * gains_g[i] / gains_g[0]
        print(f"  Gen #{i+1}: {gains_g[i]:>9.0f} ({pct_total:.1f}% cumulative, "
              f"{retention:.1f}% of Gen #1's gain)")

## 11. Alternative decisions vs. data-driven greedy

We compare our prediction-based greedy against three naive baselines that don't use the model:

| Baseline | Selection rule |
|---|---|
| Top-5 by population | Pick 5 most populous counties, 1 generator each |
| Top-5 by historical mean | Pick 5 counties with highest training-set average outage |
| Random uniform | Pick 5 counties at random (averaged over 200 trials) |
| **Our (data-driven)** | **Greedy on 48h prediction, doubling allowed** |

**Quantification:** for each strategy, report mitigation against `base_blended` (our 48h forecast).
Our greedy is optimal *for that forecast*; baselines lose value by ignoring the spatial-temporal
pattern of the upcoming storm and (importantly) by not exploiting capacity-doubling at high-impact
counties.


### 11.1 Setup — load population (`tracked`) and historical mean from `train.nc`

If `train.nc` is unavailable (e.g., running this notebook standalone), falls back to a
hard-coded 2020 Census Michigan county population dictionary.


In [ ]:
# Hardcoded 2020 Census Michigan county populations (used if train.nc unavailable)
MI_COUNTY_POPULATION_FALLBACK = {
    '26001': 10942,    '26003': 9233,     '26005': 117973,   '26007': 14694,
    '26009': 24481,    '26011': 8200,     '26013': 8438,     '26015': 27873,
    '26017': 100905,   '26019': 16574,    '26021': 154316,   '26023': 43504,
    '26025': 134371,   '26027': 49087,    '26029': 26054,    '26031': 23432,
    '26033': 36785,    '26035': 12669,    '26037': 79595,    '26039': 8351,
    '26041': 36392,    '26043': 16666,    '26045': 109094,   '26047': 33809,
    '26049': 406211,   '26051': 8758,     '26053': 24029,    '26055': 95255,
    '26057': 41202,    '26059': 45079,    '26061': 38094,    '26063': 31314,
    '26065': 284900,   '26067': 71629,    '26069': 22784,    '26071': 11355,
    '26073': 79145,    '26075': 160366,   '26077': 261670,   '26079': 16924,
    '26081': 657974,   '26083': 2046,     '26085': 11407,    '26087': 99423,
    '26089': 22042,    '26091': 99423,    '26093': 193866,   '26095': 6240,
    '26097': 23734,    '26099': 881217,   '26101': 24230,    '26103': 64054,
    '26105': 25027,    '26107': 39820,    '26109': 38040,    '26111': 41748,
    '26113': 17285,    '26115': 152021,   '26117': 64652,    '26119': 8702,
    '26121': 175824,   '26123': 41122,    '26125': 1274395,  '26127': 26102,
    '26129': 21311,    '26131': 5719,     '26133': 21077,    '26135': 8657,
    '26137': 22950,    '26139': 296200,   '26141': 13145,    '26143': 21068,
    '26145': 190124,   '26147': 160383,   '26149': 60938,    '26151': 41095,
    '26153': 6592,     '26155': 86413,    '26157': 53683,    '26159': 75677,
    '26161': 372258,   '26163': 1793561,  '26165': 32503,
}

county_population = None
hist_mean_outage  = None
TRAIN_CANDIDATES = [
    'data/train.nc',
    'drive/MyDrive/train.nc',
    '/content/drive/MyDrive/MLPS_Final_Project/data/train.nc',
]

try:
    import xarray as xr
    for p in TRAIN_CANDIDATES:
        if not os.path.exists(p):
            continue
        ds_train = xr.open_dataset(p)
        ds_locs = [str(l) for l in ds_train.location.values]

        if 'tracked' in ds_train.variables:
            tracked_arr = ds_train.tracked.transpose('timestamp', 'location').values
            # Use max over time as static population proxy
            county_population = {ds_locs[i]: float(np.nanmax(tracked_arr[:, i]))
                                 for i in range(len(ds_locs))}
            print(f"[train.nc] Loaded county_population from `tracked` field ({p})")

        if 'out' in ds_train.variables:
            out_arr = ds_train.out.transpose('timestamp', 'location').values
            hist_mean_outage = {ds_locs[i]: float(np.nanmean(out_arr[:, i]))
                                for i in range(len(ds_locs))}
            print(f"[train.nc] Loaded hist_mean_outage from `out` field ({p})")
        break
except ImportError:
    print("[skip] xarray not installed")

if county_population is None:
    county_population = {fips: MI_COUNTY_POPULATION_FALLBACK.get(fips, 25000)
                         for fips in locs}
    print("[fallback] Using hardcoded MI 2020 Census populations")

if hist_mean_outage is None:
    print("[skip] historical mean baseline unavailable (no train.nc)")

# Sanity: top 5 by population
top5_pop = sorted(((f, county_population.get(f, 0)) for f in locs),
                  key=lambda x: -x[1])[:5]
print(f"\nTop 5 by population: {[(f, int(p)) for f, p in top5_pop]}")

### 11.2 Run baselines and compare


In [ ]:
def picks_top5_by_metric(metric_dict, locs):
    """Top 5 different counties by metric, one generator each (no doubling)."""
    ranked = sorted(((f, metric_dict.get(f, 0)) for f in locs), key=lambda x: -x[1])
    fips5 = [f for f, _ in ranked][:5]
    return [locs.index(f) for f in fips5]

def picks_random_avg(locs, n_trials=200, n_gen=NUM_GENERATORS, rng=None):
    """Average mitigation over n_trials random allocations (5 distinct counties)."""
    if rng is None:
        rng = np.random.default_rng(0)
    L_ = len(locs)
    mits = []
    all_picks = []
    for _ in range(n_trials):
        picks = list(rng.choice(L_, n_gen, replace=False))
        mits.append(mitigation_total(base_blended, picks))
        all_picks.append([locs[p] for p in picks])
    avg_mit = float(np.mean(mits))
    # Use the *average mitigation* for reporting; pick a representative trial for `picks` field
    return all_picks, avg_mit, float(np.std(mits))

# Build strategy dict
strategies = {}
strategies['Top-5 by population']      = picks_top5_by_metric(county_population, locs)
if hist_mean_outage is not None:
    strategies['Top-5 by historical mean'] = picks_top5_by_metric(hist_mean_outage, locs)
strategies['Our decision (data-driven)'] = baseline_picks

# Evaluate deterministic strategies against base_blended
total_pred = base_blended.sum()
rows = []
for name, picks in strategies.items():
    mit  = mitigation_total(base_blended, picks)
    pct  = 100 * mit / total_pred
    fips = [locs[p] for p in picks]
    rows.append({'strategy': name, 'picks': fips,
                 'mitigation': mit, 'mit_%': pct})

# Add random baseline (averaged)
rand_picks_list, rand_mean_mit, rand_std = picks_random_avg(locs, n_trials=200, rng=RNG)
rows.append({
    'strategy': 'Random uniform (avg of 200)',
    'picks': f'<200 random trials, mean ± std>',
    'mitigation': rand_mean_mit, 'mit_%': 100 * rand_mean_mit / total_pred,
})

result_df = pd.DataFrame(rows)
result_df.to_csv('results/baseline_comparison.csv', index=False)

# Compute % improvement of data-driven over each baseline
our_mit = result_df.loc[result_df['strategy'] == 'Our decision (data-driven)',
                         'mitigation'].iloc[0]
result_df['gain_vs_baseline_%'] = 100 * (our_mit - result_df['mitigation']) / our_mit

print(result_df[['strategy', 'mitigation', 'mit_%', 'gain_vs_baseline_%']]
      .round(2).to_string(index=False))
print(f"\nTotal predicted outage: {int(total_pred):,}")
print(f"Random baseline mit ± std: {rand_mean_mit:.0f} ± {rand_std:.0f}")
print(f"Our mitigation:           {int(our_mit):,} ({100*our_mit/total_pred:.1f}%)")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(11, max(4, 0.7 * len(result_df))))
df_sorted = result_df.sort_values('mitigation').reset_index(drop=True)
colors = ['#1565C0' if 'data-driven' in n else '#95a5a6'
          for n in df_sorted['strategy']]
bars = ax.barh(df_sorted['strategy'], df_sorted['mit_%'], color=colors)
for bar, m_val, pct in zip(bars, df_sorted['mitigation'], df_sorted['mit_%']):
    ax.text(pct + 0.3, bar.get_y() + bar.get_height()/2,
            f"{int(m_val):,} ({pct:.1f}%)",
            va='center', fontsize=9)
ax.set_xlabel('Mitigation as % of total predicted outage')
ax.set_title('Allocation strategies — mitigation against our 48h forecast\n'
             f'(Total predicted outage: {int(total_pred):,} outage-hours)')
ax.axvline(0, color='black', linewidth=0.6)
ax.set_xlim(0, max(df_sorted['mit_%']) * 1.18)
plt.tight_layout()
plt.savefig('results/baseline_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

# One-line summary
print("\n--- Improvement of data-driven over each baseline ---")
for _, row in result_df.iterrows():
    if 'data-driven' in row['strategy']:
        continue
    print(f"  {row['strategy']:<35s}: "
          f"+{row['gain_vs_baseline_%']:5.1f}% relative gain")

## 12. Summary — combined findings

In [ ]:
print(f"{'='*60}")
print("Sensitivity + Oracle + Baseline Analysis Summary")
print(f"{'='*60}")
print(f"Baseline picks            : {baseline_fips}")
print(f"\n[Stability — perturbation × blend grid]")
print(f"  Mean overlap (stochastic): min={mean_overlap.min():.2f}, "
      f"max={mean_overlap.max():.2f}, overall={mean_overlap.mean():.2f} / 5")
print(f"  Uniform scaling overlap  : min={uniform_overlap.min()}, "
      f"max={uniform_overlap.max()} / 5")
robust = freq_df[freq_df['freq_%'] > 80]['FIPS'].tolist()
print(f"  Robust picks (>80% freq) : {robust}")
print(f"\n[Oracle — synthetic regret]")
print(f"  Mean regret              : {regret_df['regret_%'].mean():.2f}%")
print(f"  Worst-case scenario      : {regret_df.loc[regret_df['regret_%'].idxmax(), 'scenario']}"
      f" ({regret_df['regret_%'].max():.2f}%)")
print(f"  Worst scenario family    : {regret_df.groupby('family')['regret_%'].mean().idxmax()}")
print(f"\n[Greedy efficiency]")
print(f"  Gen #1 captures          : {100*gains_g[0]/total_predicted:.1f}% of total predicted")
print(f"  Top 5 gens capture       : {100*cumulative[-1]/total_predicted:.1f}% of total predicted")
print(f"  Marginal decay (#5/#1)   : {100*gains_g[-1]/gains_g[0]:.1f}%")
print(f"\n[Data-driven advantage over baselines]")
for _, row in result_df.iterrows():
    if 'data-driven' in row['strategy']:
        continue
    print(f"  vs {row['strategy']:<32s}: +{row['gain_vs_baseline_%']:.1f}% mitigation")